In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_groq import ChatGroq
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool

import requests
import random

In [3]:
load_dotenv() 
model = ChatGroq(model="openai/gpt-oss-safeguard-20b", temperature=0, max_tokens=1024)

In [4]:
search_tool = DuckDuckGoSearchRun(region="us-en")

In [6]:
@tool
def calculator(first_num: float, second_num: float, operation: str) -> dict:
    
     """Perform add, sub, mul, or div on two numbers."""
     try:
        if operation == "add":
            result = first_num + second_num
        elif operation == "sub":
            result = first_num - second_num
        elif operation == "mul":
            result = first_num * second_num
        elif operation == "div":
            if second_num == 0:
                return {"error": "Division by zero is not allowed"}
            result = first_num / second_num
        else:
            return {"error": f"Unsupported operation '{operation}'"}
        
        return {"first_num": first_num, "second_num": second_num, "operation": operation, "result": result}
     except Exception as e:
        return {"error": str(e)}

In [9]:
@tool
def get_stock_price(symbol: str) -> dict:
    """
    Fetch latest stock price for a given symbol (e.g. 'AAPL', 'TSLA') 
    using Yahoo Finance URL without requiring an API key.
    """
    try:
        url = f"https://query1.finance.yahoo.com/v7/finance/quote?symbols={symbol}"
        r = requests.get(url)
        data = r.json()
        if "quoteResponse" in data and data["quoteResponse"]["result"]:
            result = data["quoteResponse"]["result"][0]
            price = result.get("regularMarketPrice")
            return {"symbol": symbol, "price": price}
        else:
            return {"error": f"No data found for {symbol}"}
    except Exception as e:
        return {"error": str(e)}

In [10]:
tools = [get_stock_price, search_tool, calculator]

llm_with_tools = model.bind_tools(tools)

In [12]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [13]:
def chat_node(state: ChatState):
    """LLM node that may answer or request a tool call."""
    messages = state['messages']
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

tool_node = ToolNode(tools) 

In [14]:
graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)